In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -U "transformers==4.44.2" "accelerate==0.34.2" "peft==0.11.1" optuna evaluate rouge_score

In [ ]:
from huggingface_hub import login

login("your_token")

In [ ]:
import pandas as pd
from transformers import AutoTokenizer
from tqdm import tqdm

# ===== CONFIG =====
CSV_PATH = "/kaggle/input/project-multi/test_random_600-2.csv"
TEXT_COL = "description_html_clean"
MODEL = "google/flan-t5-xl"   # đổi sang t5-large, facebook/bart-large, gemma, llama…
MAX_SOURCE_LEN = 512

# ===== LOAD DATA =====
df = pd.read_csv(CSV_PATH)
df = df[df[TEXT_COL].notna()]   # bỏ NaN
texts = df[TEXT_COL].tolist()

# ===== LOAD TOKENIZER =====
tokenizer = AutoTokenizer.from_pretrained(MODEL)

token_lengths = []
overflow_count = 0

for txt in tqdm(texts, desc="Tokenizing"):
    toks = tokenizer.encode(txt, add_special_tokens=True)
    L = len(toks)
    token_lengths.append(L)
    if L > MAX_SOURCE_LEN:
        overflow_count += 1

total = len(token_lengths)
pct_overflow = overflow_count / total * 100
avg_len = sum(token_lengths) / total

print(f"Model tokenizer: {MODEL}")
print(f"Total samples: {total}")
print(f"Average token length: {avg_len:.2f}")
print(f"Max token length: {max(token_lengths)}")
print(f"Min token length: {min(token_lengths)}")
print(f"% samples > {MAX_SOURCE_LEN} tokens: {pct_overflow:.2f}%")